<a href="https://colab.research.google.com/github/nohaelkachach/casiav2-splicing-gradcam-audit/blob/main/notebooks/04_gradcam_iou.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Grad-CAM generation and IoU computation

Generates Grad-CAM heatmaps for the 1,828 validated spliced images using both trained classifiers (ResNet18, EfficientNet-B0), then computes two localization-faithfulness measures against each ground-truth mask: single-threshold IoU (the primary outcome variable Y for the moderation regression in notebook 05) and AUC-IoU (a threshold-free robustness check).

**Design decisions stated explicitly:**
- **Target class**: Grad-CAM is always computed for the fixed "Tampered" class (class 1), regardless of what the classifier actually predicted for a given image. This studies the model's Tampered-class evidence as a property of the (known) ground truth, rather than conditioning on whether the classifier happened to predict correctly for that instance.
- **Target layer**: the final convolutional block of each architecture (`layer4` for ResNet18, `features[-1]` for EfficientNet-B0) — the standard choice in the original Grad-CAM paper and in prior forensic-CV Grad-CAM work.
- **CAM binarization threshold**: the primary IoU measure binarizes each heatmap at its own mean activation value (a dataset-adaptive choice, not a fixed global threshold like 0.5, since heatmaps vary in how "peaked" they are).
- **Why also compute AUC-IoU**: single-threshold IoU is known to be sensitive to the specific threshold chosen — Aksoy (2025) shows switching thresholds can reorder attribution-method rankings and shift size-stratified scores by over 100 percentage points in some cases. AUC-IoU sweeps a range of thresholds and integrates IoU across all of them, giving a threshold-free measure. Reporting both lets us confirm the size-moderated polarity effect found with single-threshold IoU is not an artifact of the specific threshold used.

## Setup — mount Drive, rebuild validated file lists

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image

base = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised"  # adjust to your actual path
tp_dir = os.path.join(base, "Tp")
gt_dir = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_Groundtruth/CASIA2.0_Groundtruth"  # adjust if the real path differs

with open(os.path.join(base, "tp_list.txt")) as f:
    tp_list_content = [line.strip() for line in f if line.strip()]
tp_files_final = sorted(set(tp_list_content) & set(os.listdir(tp_dir)))
spliced_files = [f for f in tp_files_final if f.split("_")[1] == "D"]
print(f"Spliced-only file count: {len(spliced_files)}")  # expect 1828

def get_mask_path(image_filename):
    stem = os.path.splitext(image_filename)[0]
    return os.path.join(gt_dir, f"{stem}_gt.png")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Mounted at /content/drive
Spliced-only file count: 1828
Device: cuda


## Load trained classifiers
Loads the best checkpoints from notebook 03. Both must exist on Drive before running this notebook.

In [2]:
resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_resnet_best.pt", map_location=device))
resnet = resnet.to(device)
resnet.eval()

effnet = models.efficientnet_b0(weights=None)
num_features = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(num_features, 2)
effnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_effnet_best.pt", map_location=device))
effnet = effnet.to(device)
effnet.eval()

print("Both checkpoints loaded successfully.")

Both checkpoints loaded successfully.


## Grad-CAM implementation (forward/backward hooks)
Captures activations of the target layer on the forward pass and gradients of the target class score on the backward pass. Channel-wise gradients are global-average-pooled to give one importance weight per channel; the weighted sum of channels, after ReLU, is the heatmap.

In [3]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class):
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[0, target_class]
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam

gradcam_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## IoU computation — single-threshold (primary) and AUC-IoU (robustness check)
`compute_iou` resizes the CAM up to the mask's resolution and binarizes it at its own mean activation. `compute_auc_iou` sweeps thresholds from 0.1 to 0.9 and integrates IoU across them (trapezoidal rule), giving a threshold-free measure per Aksoy (2025).

In [4]:
def _resize_cam(cam, gt_mask_binary):
    return np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize(
        (gt_mask_binary.shape[1], gt_mask_binary.shape[0]), Image.BILINEAR
    )) / 255.0

def compute_iou(cam_resized, gt_mask_binary):
    thresh = cam_resized.mean()
    cam_binary = cam_resized > thresh

    intersection = np.logical_and(cam_binary, gt_mask_binary).sum()
    union = np.logical_or(cam_binary, gt_mask_binary).sum()

    if union == 0:
        return None

    return intersection / union

def compute_auc_iou(cam_resized, gt_mask_binary, thresholds=np.arange(0.1, 1.0, 0.1)):
    ious = []
    for t in thresholds:
        cam_binary = cam_resized > t
        intersection = np.logical_and(cam_binary, gt_mask_binary).sum()
        union = np.logical_or(cam_binary, gt_mask_binary).sum()
        ious.append(intersection / union if union > 0 else 0.0)

    return np.trapz(ious, thresholds)

## Run Grad-CAM + IoU for a given model
Shared routine for both architectures below — target class is always 1 (Tampered), per the design decision stated at the top of this notebook.

In [5]:
def run_gradcam_iou(model, target_layer, model_label):
    gradcam = GradCAM(model, target_layer)
    records = []

    for i, fname in enumerate(spliced_files):
        img_path = os.path.join(tp_dir, fname)
        mask_path = get_mask_path(fname)

        if not os.path.exists(mask_path):
            continue

        img_pil = Image.open(img_path).convert("RGB")
        input_tensor = gradcam_transform(img_pil).unsqueeze(0).to(device)

        cam = gradcam.generate(input_tensor, target_class=1)

        orig_size = img_pil.size
        mask_pil = Image.open(mask_path).convert("L").resize(orig_size, Image.NEAREST)
        gt_mask_binary = np.array(mask_pil) > 127

        cam_resized = _resize_cam(cam, gt_mask_binary)
        iou = compute_iou(cam_resized, gt_mask_binary)
        if iou is None:
            continue
        auc_iou = compute_auc_iou(cam_resized, gt_mask_binary)

        records.append({"filename": fname, "iou": iou, "auc_iou": auc_iou})

        if (i + 1) % 200 == 0:
            print(f"[{model_label}] Processed {i+1}/{len(spliced_files)}")

    iou_df = pd.DataFrame(records)
    print(f"\n[{model_label}] Done. Computed IoU for {len(iou_df)} of {len(spliced_files)} spliced images.")
    print(iou_df.describe())
    return iou_df

## Run for ResNet18

In [6]:
iou_df_resnet = run_gradcam_iou(resnet, resnet.layer4[-1], "ResNet18")
iou_df_resnet.to_csv("/content/drive/MyDrive/CASIA2.0/gradcam_iou_resnet.csv", index=False)

/tmp/ipykernel_1188/2463263135.py:26: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(ious, thresholds)


[ResNet18] Processed 200/1828
[ResNet18] Processed 400/1828
[ResNet18] Processed 600/1828
[ResNet18] Processed 800/1828
[ResNet18] Processed 1000/1828
[ResNet18] Processed 1200/1828
[ResNet18] Processed 1400/1828
[ResNet18] Processed 1600/1828
[ResNet18] Processed 1800/1828

[ResNet18] Done. Computed IoU for 1828 of 1828 spliced images.
               iou      auc_iou
count  1828.000000  1828.000000
mean      0.147837     0.111535
std       0.146779     0.094881
min       0.000000     0.000000
25%       0.030283     0.026364
50%       0.095382     0.092835
75%       0.226362     0.181224
max       0.778294     0.478033


## Run for EfficientNet-B0
Target layer is `features[-1]` — EfficientNet's final conv block — not `layer4`, which is ResNet-specific.

In [7]:
iou_df_effnet = run_gradcam_iou(effnet, effnet.features[-1], "EfficientNet-B0")
iou_df_effnet.to_csv("/content/drive/MyDrive/CASIA2.0/gradcam_iou_effnet.csv", index=False)

/tmp/ipykernel_1188/2463263135.py:26: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(ious, thresholds)


[EfficientNet-B0] Processed 200/1828
[EfficientNet-B0] Processed 400/1828
[EfficientNet-B0] Processed 600/1828
[EfficientNet-B0] Processed 800/1828
[EfficientNet-B0] Processed 1000/1828
[EfficientNet-B0] Processed 1200/1828
[EfficientNet-B0] Processed 1400/1828
[EfficientNet-B0] Processed 1600/1828
[EfficientNet-B0] Processed 1800/1828

[EfficientNet-B0] Done. Computed IoU for 1828 of 1828 spliced images.
               iou      auc_iou
count  1828.000000  1828.000000
mean      0.162065     0.124369
std       0.158252     0.098071
min       0.000000     0.000000
25%       0.035463     0.034481
50%       0.110952     0.114582
75%       0.245977     0.193429
max       0.891132     0.521208


## Merge with extracted features (notebook 02) for the regression
Produces the two regression-ready datasets used in notebook 05. Each carries both `iou` (primary outcome) and `auc_iou` (threshold-free robustness check) alongside splice_size_frac, polarity, and abs_contrast.

In [8]:
features_df = pd.read_csv("/content/drive/MyDrive/CASIA2.0/splice_features.csv")

merged_resnet = features_df.merge(iou_df_resnet, on="filename", how="inner")
merged_resnet.to_csv("/content/drive/MyDrive/CASIA2.0/regression_ready_resnet.csv", index=False)
print(f"ResNet18 regression-ready dataset: {len(merged_resnet)} rows")

merged_effnet = features_df.merge(iou_df_effnet, on="filename", how="inner")
merged_effnet.to_csv("/content/drive/MyDrive/CASIA2.0/regression_ready_effnet.csv", index=False)
print(f"EfficientNet-B0 regression-ready dataset: {len(merged_effnet)} rows")

ResNet18 regression-ready dataset: 1828 rows
EfficientNet-B0 regression-ready dataset: 1828 rows
